# Mono-Graph vs Micro-Graphs
Companion notebook for the **Mono-Graph vs Micro-Graphs** section of Chapter 12.

**Mono-graph**: one service, one codebase, fewer network hops, simpler tracing. **Micro-graph**: split by real ownership/scaling/isolation boundaries -- at the cost of more network calls, auth, retries, and version coordination. The book gives no code here, just the two flow diagrams -- this notebook makes that latency/hop tradeoff concrete with a tiny simulation.

A mono-graph: one process, one function per step, no network calls between them:

In [1]:
import time


def retriever_step(request: dict) -> dict:
    return {**request, "context": ["policy chunk"]}


def tool_step(request: dict) -> dict:
    return {**request, "tool_result": {"status": "ok"}}


def reviewer_step(request: dict) -> dict:
    return {**request, "approved": True}


def run_mono_graph(request: dict) -> dict:
    """client -> graph-service -> retriever/tool/reviewer nodes -> response"""
    request = retriever_step(request)
    request = tool_step(request)
    request = reviewer_step(request)
    return request

A micro-graph: the same three steps, but each one is a separate "service" reached over a simulated network hop (a small fixed delay stands in for the real network cost):

In [2]:
NETWORK_HOP_S = 0.02  # a small fixed delay standing in for a real network round trip


def call_service(step_fn, request: dict) -> dict:
    time.sleep(NETWORK_HOP_S)  # the cost a mono-graph does not pay
    return step_fn(request)


def run_micro_graph(request: dict) -> dict:
    """client -> intake-graph -> planning-graph -> execution-graph -> review-graph"""
    request = call_service(retriever_step, request)
    request = call_service(tool_step, request)
    request = call_service(reviewer_step, request)
    return request

In [3]:
request = {"question": "Can this refund be approved?"}

t0 = time.perf_counter()
mono_result = run_mono_graph(request)
mono_ms = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
micro_result = run_micro_graph(request)
micro_ms = (time.perf_counter() - t0) * 1000

print(f"mono-graph:  {mono_ms:.1f} ms, same final result: {mono_result == micro_result}")
print(f"micro-graph: {micro_ms:.1f} ms ({micro_ms - mono_ms:.1f} ms added by 3 network hops)")

mono-graph:  0.0 ms, same final result: True
micro-graph: 61.3 ms (61.2 ms added by 3 network hops)


Same three steps, same final answer -- the micro-graph is strictly slower here because it paid for network hops the mono-graph never needed. That's the tradeoff the book describes in prose: split when there's a *real* reason (different ownership, different scaling needs, stronger isolation), not because splitting sounds cleaner. If your planner is CPU-light but your document processor is heavy, the ability to scale them independently can easily be worth this added latency -- but it's a cost you're paying on purpose, not a free upgrade.